# Cell 1 - Setup & Dependencies

This sandbox notebook evaluates whether a single Claude Haiku 4.5 tool call can extract structured antique US hunting and fishing license fields from test images, then map those outputs onto Hunt's reference CSVs with lightweight fuzzy resolution.

Guardrails:
- Pure Python only. No Django imports and no ORM calls.
- Reference data loads from `utilities/cleaned/*.csv` with raw `utilities/ref_data/*.csv` fallback.
- Test images are discovered from `media/listings/` and `media/collections/` with globbing.

run:
- jupyter lab sandbox/image_prefill_sandbox.ipynb


In [ ]:
%pip install anthropic python-dotenv pillow rapidfuzz pandas


: 

In [ ]:
import base64
import glob
import os
import re
import time
from collections import defaultdict
from html import escape
from io import BytesIO
from pathlib import Path
from typing import Any

import anthropic
import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, display
from PIL import Image
from rapidfuzz import fuzz, process


def find_project_root(start: Path | None = None) -> Path:
    '''Walk upward until the repo root markers are found.'''
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        markers = ['apps', 'utilities', 'media', 'sandbox']
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    return current


PROJECT_ROOT = find_project_root()
ENV_PATH = PROJECT_ROOT / '.env'
MEDIA_ROOT = PROJECT_ROOT / 'media'
LISTINGS_IMAGE_DIR = MEDIA_ROOT / 'listings'
COLLECTIONS_IMAGE_DIR = MEDIA_ROOT / 'collections'
CLEANED_DATA_DIR = PROJECT_ROOT / 'utilities' / 'cleaned'
RAW_DATA_DIR = PROJECT_ROOT / 'utilities' / 'ref_data'
COMMON_IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
MODEL_ID = 'claude-haiku-4-5'
HAIKU_INPUT_PRICE_PER_MILLION_USD = 1.0
HAIKU_OUTPUT_PRICE_PER_MILLION_USD = 5.0
RESAMPLE_LANCZOS = Image.Resampling.LANCZOS if hasattr(Image, 'Resampling') else Image.LANCZOS

load_dotenv(ENV_PATH)
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print(f'Project root: {PROJECT_ROOT}')
if client is None:
    print('ANTHROPIC_API_KEY not found.')
    print('Set it in the environment or in a repo-root .env file, for example:')
    print('ANTHROPIC_API_KEY=your_key_here')
else:
    print(f'Anthropic client ready for model: {MODEL_ID}')


## Cell 2 - Load Reference Data

These helpers normalize the cleaned CSVs into in-memory lookup structures. If the cleaned files are missing, the cell falls back to the raw `ref_data` CSVs and reshapes them into the same matching format.


In [ ]:
def normalize_state_abbrev(value: Any) -> str:
    '''Normalize state-like codes and treat missing values as universal.'''
    if pd.isna(value):
        return ''
    text = str(value).strip().upper()
    return '' if text in {'NAN', 'NONE'} else text


def normalize_lookup_key(value: Any) -> str:
    '''Normalize lookup text for exact state matching.'''
    text = str(value or '').strip()
    text = re.sub(r'\.', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.upper()


def coerce_bool(value: Any) -> bool:
    '''Convert mixed CSV boolean encodings into Python booleans.'''
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {'true', '1', 'yes', 'y'}


def make_slug(*parts: Any) -> str:
    '''Build a lightweight slug for raw fallback rows.'''
    pieces: list[str] = []
    for part in parts:
        if part is None:
            continue
        text = str(part).strip()
        if text and text.lower() != 'nan':
            pieces.append(text)
    raw = '-'.join(pieces)
    cleaned = re.sub(r'[^a-z0-9]+', '-', raw.lower())
    return cleaned.strip('-')


def build_raw_license_types() -> pd.DataFrame:
    '''Normalize raw license class and add-on CSVs into cleaned-style rows.'''
    classes_df = pd.read_csv(RAW_DATA_DIR / 'license_classes.csv')
    addons_df = pd.read_csv(RAW_DATA_DIR / 'addons_permits.csv')
    rows: list[dict[str, Any]] = []
    class_columns = {
        'residency': 'residency',
        'holder_eligibility': 'holder_eligibility',
        'activity_scope': 'activity_scope',
        'duration': 'duration',
    }
    for row in classes_df.to_dict('records'):
        state_abbrev = normalize_state_abbrev(row.get('state_abbrev'))
        for category, column in class_columns.items():
            name = str(row.get(column) or '').strip()
            if name:
                rows.append({
                    'state_abbrev': state_abbrev,
                    'name': name,
                    'category': category,
                    'slug': make_slug(state_abbrev or 'universal', category, name),
                    'is_system_value': True,
                })
    for row in addons_df.to_dict('records'):
        state_abbrev = normalize_state_abbrev(row.get('state_abbrev'))
        state_abbrev = 'FD' if state_abbrev == 'FEDERAL' else state_abbrev
        name = str(row.get('addon_name') or '').strip()
        if name:
            rows.append({
                'state_abbrev': state_abbrev,
                'name': name,
                'category': 'addon_type',
                'slug': make_slug(state_abbrev or 'universal', 'addon_type', name),
                'is_system_value': True,
            })
    enum_rows = []
    for category in ['residency', 'holder_eligibility', 'activity_scope', 'duration', 'addon_type']:
        enum_rows.append({
            'state_abbrev': '',
            'name': 'Other',
            'category': category,
            'slug': make_slug('universal', category, 'other'),
            'is_system_value': True,
        })
    for name in ['Paper/Cardstock', 'Metal Button', 'Metal Tag', 'Celluloid', 'Fabric/Canvas', 'Plastic', 'Other']:
        enum_rows.append({
            'state_abbrev': '',
            'name': name,
            'category': 'material',
            'slug': make_slug('universal', 'material', name),
            'is_system_value': True,
        })
    for name in ['Rectangle', 'Square', 'Button/Disc', 'Tag (with hole)', 'Strip', 'Irregular/Custom', 'Other']:
        enum_rows.append({
            'state_abbrev': '',
            'name': name,
            'category': 'shape',
            'slug': make_slug('universal', 'shape', name),
            'is_system_value': True,
        })
    license_df = pd.DataFrame(rows + enum_rows)
    return license_df.drop_duplicates(subset=['state_abbrev', 'name', 'category']).reset_index(drop=True)


In [ ]:
def load_reference_data() -> dict[str, Any]:
    '''Load reference CSVs into structures used by the sandbox resolver.'''
    cleaned_paths = {
        'states': CLEANED_DATA_DIR / 'states.csv',
        'geographic_units': CLEANED_DATA_DIR / 'geographic_units.csv',
        'license_types': CLEANED_DATA_DIR / 'license_types.csv',
    }
    using_cleaned = all(path.exists() for path in cleaned_paths.values())
    notes: list[str] = []
    if using_cleaned:
        states_df = pd.read_csv(cleaned_paths['states'])
        geo_df = pd.read_csv(cleaned_paths['geographic_units'])
        license_df = pd.read_csv(cleaned_paths['license_types'])
    else:
        notes.append('Cleaned CSVs missing; falling back to utilities/ref_data/*.csv.')
        states_df = pd.read_csv(RAW_DATA_DIR / 'states.csv')
        states_df['slug'] = states_df['state_name'].map(make_slug)
        geo_df = pd.read_csv(RAW_DATA_DIR / 'geographic_units.csv').rename(columns={'unit_name': 'name'})
        geo_df['slug'] = geo_df.apply(lambda row: make_slug(row['state_abbrev'], row['name']), axis=1)
        license_df = build_raw_license_types()

    states_df['state_abbrev'] = states_df['state_abbrev'].map(normalize_state_abbrev)
    geo_df['state_abbrev'] = geo_df['state_abbrev'].map(normalize_state_abbrev)
    license_df['state_abbrev'] = license_df['state_abbrev'].map(normalize_state_abbrev)

    states = {'by_name': {}, 'by_abbrev': {}, 'records': []}
    for row in states_df.to_dict('records'):
        record = {
            'name': str(row['state_name']).strip(),
            'abbrev': str(row['state_abbrev']).strip(),
            'slug': str(row.get('slug') or make_slug(row['state_name'])).strip(),
        }
        states['records'].append(record)
        states['by_name'][normalize_lookup_key(record['name'])] = record
        states['by_abbrev'][normalize_lookup_key(record['abbrev'])] = record

    geographic_units: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in geo_df.to_dict('records'):
        geographic_units[row['state_abbrev']].append({
            'name': str(row['name']).strip(),
            'unit_type': str(row.get('unit_type') or '').strip(),
            'is_statewide': coerce_bool(row.get('is_statewide')),
            'slug': str(row.get('slug') or make_slug(row['state_abbrev'], row['name'])).strip(),
        })

    license_types: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
    for row in license_df.to_dict('records'):
        license_types[(row['state_abbrev'], str(row['category']).strip())].append({
            'name': str(row['name']).strip(),
            'slug': str(row.get('slug') or make_slug(row['state_abbrev'], row['category'], row['name'])).strip(),
            'category': str(row['category']).strip(),
            'state_abbrev': row['state_abbrev'],
        })

    counts = {
        'states': len(states['records']),
        'geographic_units': sum(len(items) for items in geographic_units.values()),
        'license_types': sum(len(items) for items in license_types.values()),
    }
    return {
        'states': states,
        'geographic_units': dict(geographic_units),
        'license_types': dict(license_types),
        'source_label': 'utilities/cleaned/*.csv' if using_cleaned else 'utilities/ref_data/*.csv',
        'notes': notes,
        'counts': counts,
    }


reference_data = load_reference_data()
print(f"Reference source: {reference_data['source_label']}")
for note in reference_data['notes']:
    print(note)
print(
    f"Loaded {reference_data['counts']['states']} states, "
    f"{reference_data['counts']['geographic_units']} geographic units, "
    f"{reference_data['counts']['license_types']} license types."
)


## Cell 3 - Image Helpers

The VLM helper resizes each image so the longest edge is at most 1568px, flattens transparency onto white, and re-encodes to JPEG before base64 upload. The image discovery helper globs both sandbox media folders instead of hardcoding filenames.


In [ ]:
def load_image_bytes(path: str | Path) -> tuple[bytes, str, str]:
    '''Resize an image for VLM submission and return JPEG bytes plus base64.'''
    image_path = Path(path)
    with Image.open(image_path) as img:
        image = img.copy()
    image.thumbnail((1568, 1568), RESAMPLE_LANCZOS)
    if image.mode in {'RGBA', 'LA'} or (image.mode == 'P' and 'transparency' in image.info):
        rgba = image.convert('RGBA')
        background = Image.new('RGB', rgba.size, 'white')
        background.paste(rgba, mask=rgba.getchannel('A'))
        image = background
    else:
        image = image.convert('RGB')
    buffer = BytesIO()
    image.save(buffer, format='JPEG', quality=90, optimize=True)
    image_bytes = buffer.getvalue()
    image_b64 = base64.b64encode(image_bytes).decode('utf-8')
    return image_bytes, image_b64, 'image/jpeg'


def discover_test_images() -> list[tuple[str, Path]]:
    '''Discover listing and collection sandbox images using glob patterns.'''
    image_sets: list[tuple[str, Path]] = []
    for source_form, folder in [('listing', LISTINGS_IMAGE_DIR), ('collection', COLLECTIONS_IMAGE_DIR)]:
        for matched_path in sorted(glob.glob(str(folder / '*'))):
            candidate = Path(matched_path)
            if candidate.suffix.lower() in COMMON_IMAGE_EXTENSIONS:
                image_sets.append((source_form, candidate))
    return image_sets


preview_images = tuple(discover_test_images())
print(f'Discovered {len(preview_images)} test images.')
for source_form, path in preview_images:
    print(f'- {source_form}: {path.name}')


## Cell 4 - Extraction Schema & Prompt

System prompt draft:

```text
You are an expert cataloger of antique US hunting and fishing licenses.

Work only from the visible image. Be strictly honest:
- Return null for anything not clearly legible or visually supported.
- Never guess.
- Never hallucinate serial numbers.
- Transcribe serial numbers character-by-character exactly as shown.
- raw_text_transcription should include every legible word, number, mark, or abbreviation on the item even if it does not map to a structured field.
- Use per_field_confidence as numeric values between 0.0 and 1.0 keyed by the exact field names you populated.

Confidence guide:
- 0.90 to 1.00 = certain from the image.
- 0.60 to 0.89 = partially legible or lightly inferred from strong context.
- Below 0.60 = a guess, so prefer null instead.

If the item is not an antique US hunting or fishing license, still transcribe what is legible and leave unsupported fields null.
```


In [ ]:
EXTRACTION_TOOL = {
    'name': 'extract_license_fields',
    'description': 'Extract structured fields from an antique US hunting or fishing license image.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'state_name_or_abbrev': {'type': ['string', 'null']},
            'license_year': {'type': ['integer', 'null']},
            'era_guess': {
                'type': ['string', 'null'],
                'enum': ['Pre-1920', '1920s', '1930s', '1940s', '1950s', '1960s', '1970s', '1980s', '1990s', '2000s', None],
            },
            'geographic_unit_name': {'type': ['string', 'null']},
            'is_statewide': {'type': 'boolean'},
            'residency': {'type': ['string', 'null']},
            'holder_eligibility': {'type': ['string', 'null']},
            'activity_scope': {'type': ['string', 'null']},
            'duration': {'type': ['string', 'null']},
            'addon_type': {'type': ['string', 'null']},
            'serial_number': {'type': ['string', 'null']},
            'material': {
                'type': ['string', 'null'],
                'enum': ['Paper/Cardstock', 'Metal Button', 'Metal Tag', 'Celluloid', 'Fabric/Canvas', 'Plastic', None],
            },
            'shape': {
                'type': ['string', 'null'],
                'enum': ['Rectangle', 'Square', 'Button/Disc', 'Tag (with hole)', 'Strip', 'Irregular/Custom', None],
            },
            'dominant_colors': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 3},
            'condition_visual': {
                'type': ['string', 'null'],
                'enum': ['poor', 'fair', 'good', 'very_good', 'excellent', 'mint', None],
            },
            'raw_text_transcription': {'type': 'string'},
            'per_field_confidence': {'type': 'object'},
        },
        'required': ['raw_text_transcription', 'per_field_confidence'],
    },
}


SYSTEM_PROMPT = '''You are an expert cataloger of antique US hunting and fishing licenses.

Work only from the visible image. Be strictly honest:
- Return null for anything not clearly legible or visually supported.
- Never guess.
- Never hallucinate serial numbers.
- Transcribe serial numbers character-by-character exactly as shown.
- raw_text_transcription should include every legible word, number, mark, or abbreviation on the item even if it does not map to a structured field.
- Use per_field_confidence as numeric values between 0.0 and 1.0 keyed by the exact field names you populated.

Confidence guide:
- 0.90 to 1.00 = certain from the image.
- 0.60 to 0.89 = partially legible or lightly inferred from strong context.
- Below 0.60 = a guess, so prefer null instead.

If the item is not an antique US hunting or fishing license, still transcribe what is legible and leave unsupported fields null.''' 

print(f"Tool schema ready: {EXTRACTION_TOOL['name']}")


## Cell 5 - VLM Extraction Function

This wrapper forces Claude to answer via the `extract_license_fields` tool, captures latency and token usage, and estimates spend using the sandbox's hardcoded Haiku pricing assumptions (`$1/M` input and `$5/M` output).


In [ ]:
def extract(image_path: str) -> dict[str, Any]:
    '''Run one Claude tool-use extraction against a license image.'''
    if client is None:
        return {
            'error': 'ANTHROPIC_API_KEY is not set. Add it to the environment or repo-root .env before running extraction.'
        }

    started = time.perf_counter()
    try:
        _, image_b64, media_type = load_image_bytes(image_path)
        response = client.messages.create(
            model=MODEL_ID,
            max_tokens=1200,
            system=SYSTEM_PROMPT,
            tools=[EXTRACTION_TOOL],
            tool_choice={'type': 'tool', 'name': EXTRACTION_TOOL['name']},
            messages=[{
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {'type': 'base64', 'media_type': media_type, 'data': image_b64},
                    },
                    {
                        'type': 'text',
                        'text': 'Extract fields from this license using the extract_license_fields tool.',
                    },
                ],
            }],
        )
        tool_block = next((block for block in response.content if getattr(block, 'type', None) == 'tool_use'), None)
        if tool_block is None:
            return {'error': f'No tool_use block returned for {image_path}.'}

        usage_block = getattr(response, 'usage', None)
        usage = {
            'input_tokens': int(getattr(usage_block, 'input_tokens', 0) or 0),
            'output_tokens': int(getattr(usage_block, 'output_tokens', 0) or 0),
            'cache_creation_input_tokens': int(getattr(usage_block, 'cache_creation_input_tokens', 0) or 0),
            'cache_read_input_tokens': int(getattr(usage_block, 'cache_read_input_tokens', 0) or 0),
        }
        total_input_tokens = usage['input_tokens'] + usage['cache_creation_input_tokens'] + usage['cache_read_input_tokens']
        cost_usd = (
            total_input_tokens * HAIKU_INPUT_PRICE_PER_MILLION_USD
            + usage['output_tokens'] * HAIKU_OUTPUT_PRICE_PER_MILLION_USD
        ) / 1_000_000
        raw_payload = tool_block.input if isinstance(tool_block.input, dict) else dict(tool_block.input)
        return {
            'raw': raw_payload,
            'usage': usage,
            'latency_ms': round((time.perf_counter() - started) * 1000, 1),
            'cost_usd': round(cost_usd, 6),
            'model': MODEL_ID,
        }
    except Exception as exc:
        return {'error': f'{type(exc).__name__}: {exc}'}


## Cell 6 - Resolver

The resolver does exact state matching, state-narrowed geographic unit matching, and per-category license type fuzzy matching with RapidFuzz `WRatio`. Non-vocabulary fields keep their original values and derive tiers from confidence alone.


In [ ]:
LICENSE_TYPE_FIELDS = ['residency', 'holder_eligibility', 'activity_scope', 'duration', 'addon_type']


def has_value(value: Any) -> bool:
    '''Treat None, blank strings, and empty lists as missing while preserving False.'''
    if value is None:
        return False
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, list):
        return len(value) > 0
    return True


def parse_confidence(raw: dict[str, Any], field: str) -> float:
    '''Extract a numeric field confidence from the VLM payload.'''
    confidence_map = raw.get('per_field_confidence') or {}
    value = confidence_map.get(field, 0)
    if isinstance(value, dict):
        value = value.get('value', value.get('confidence', value.get('score', 0)))
    try:
        return max(0.0, min(1.0, float(value)))
    except (TypeError, ValueError):
        return 0.0


def classify_tier(vlm_conf: float, match_score: int) -> str:
    '''Apply the notebook tier thresholds.'''
    if vlm_conf >= 0.85 and match_score >= 95:
        return 'high'
    if vlm_conf >= 0.65 and match_score >= 80:
        return 'medium'
    if vlm_conf >= 0.40:
        return 'low'
    return 'unmatched'


def direct_field_result(source_value: Any, vlm_conf: float, *, serial_cap: bool = False) -> dict[str, Any]:
    '''Resolve a non-vocabulary field using confidence only.'''
    match_score = 100 if has_value(source_value) else 0
    tier = classify_tier(vlm_conf, match_score) if match_score else 'unmatched'
    if serial_cap and tier in {'high', 'medium'}:
        tier = 'low'
    return {
        'value': source_value if has_value(source_value) else None,
        'source_text': source_value,
        'match_score': match_score,
        'vlm_conf': vlm_conf,
        'tier': tier,
    }


def fuzzy_match(source_text: Any, candidates: list[dict[str, Any]], vlm_conf: float) -> dict[str, Any]:
    '''Fuzzy-match text against candidate names using RapidFuzz WRatio.'''
    if not has_value(source_text):
        return {'value': None, 'source_text': source_text, 'match_score': 0, 'vlm_conf': vlm_conf, 'tier': 'unmatched'}
    if not candidates:
        return {'value': source_text, 'source_text': source_text, 'match_score': 0, 'vlm_conf': vlm_conf, 'tier': classify_tier(vlm_conf, 0)}
    names = [candidate['name'] for candidate in candidates]
    match = process.extractOne(str(source_text), names, scorer=fuzz.WRatio)
    if match is None:
        return {'value': source_text, 'source_text': source_text, 'match_score': 0, 'vlm_conf': vlm_conf, 'tier': classify_tier(vlm_conf, 0)}
    _, match_score, match_index = match
    matched = candidates[match_index]
    return {
        'value': matched['slug'] if match_score > 0 else source_text,
        'source_text': source_text,
        'match_score': int(match_score),
        'vlm_conf': vlm_conf,
        'tier': classify_tier(vlm_conf, int(match_score)),
    }


def license_candidates(reference_data: dict[str, Any], state_abbrev: str, category: str) -> list[dict[str, Any]]:
    '''Return state-specific candidates plus universal fallback values.'''
    candidates = list(reference_data['license_types'].get((state_abbrev, category), []))
    candidates.extend(reference_data['license_types'].get(('', category), []))
    if category == 'addon_type':
        candidates.extend(reference_data['license_types'].get(('FD', category), []))
    deduped: list[dict[str, Any]] = []
    seen: set[str] = set()
    for candidate in candidates:
        key = candidate['slug']
        if key not in seen:
            deduped.append(candidate)
            seen.add(key)
    return deduped


In [ ]:
def resolve(raw: dict[str, Any], reference_data: dict[str, Any]) -> dict[str, Any]:
    '''Resolve raw VLM output onto state, geography, and license-type reference data.'''
    state_text = raw.get('state_name_or_abbrev')
    state_conf = parse_confidence(raw, 'state_name_or_abbrev')
    state_record = None
    if has_value(state_text):
        state_key = normalize_lookup_key(state_text)
        state_record = reference_data['states']['by_abbrev'].get(state_key) or reference_data['states']['by_name'].get(state_key)
    state_score = 100 if state_record else 0
    state_detail = {
        'value': state_record['slug'] if state_record else state_text,
        'source_text': state_text,
        'match_score': state_score,
        'vlm_conf': state_conf,
        'tier': classify_tier(state_conf, state_score) if has_value(state_text) else 'unmatched',
    }
    state_abbrev = state_record['abbrev'] if state_record else ''

    geo_text = raw.get('geographic_unit_name')
    geo_conf = parse_confidence(raw, 'geographic_unit_name')
    is_statewide_value = raw['is_statewide'] if 'is_statewide' in raw else None
    if is_statewide_value and state_abbrev:
        statewide_row = next((item for item in reference_data['geographic_units'].get(state_abbrev, []) if item['is_statewide']), None)
        geo_detail = {
            'value': statewide_row['slug'] if statewide_row else 'Statewide',
            'source_text': geo_text or 'Statewide',
            'match_score': 100 if statewide_row else 0,
            'vlm_conf': geo_conf,
            'tier': classify_tier(geo_conf, 100 if statewide_row else 0) if has_value(geo_text) or is_statewide_value else 'unmatched',
        }
    elif state_abbrev:
        geo_candidates = [item for item in reference_data['geographic_units'].get(state_abbrev, []) if not item['is_statewide']]
        geo_detail = fuzzy_match(geo_text, geo_candidates, geo_conf)
    else:
        geo_detail = {'value': geo_text, 'source_text': geo_text, 'match_score': 0, 'vlm_conf': geo_conf, 'tier': 'unmatched'}

    fields = {
        'state': state_detail,
        'license_year': direct_field_result(raw.get('license_year'), parse_confidence(raw, 'license_year')),
        'era_guess': direct_field_result(raw.get('era_guess'), parse_confidence(raw, 'era_guess')),
        'geographic_unit': geo_detail,
        'is_statewide': direct_field_result(is_statewide_value, parse_confidence(raw, 'is_statewide')),
        'serial_number': direct_field_result(raw.get('serial_number'), parse_confidence(raw, 'serial_number'), serial_cap=True),
        'material': fuzzy_match(raw.get('material'), license_candidates(reference_data, '', 'material'), parse_confidence(raw, 'material')),
        'shape': fuzzy_match(raw.get('shape'), license_candidates(reference_data, '', 'shape'), parse_confidence(raw, 'shape')),
        'dominant_colors': direct_field_result(raw.get('dominant_colors'), parse_confidence(raw, 'dominant_colors')),
        'condition_visual': direct_field_result(raw.get('condition_visual'), parse_confidence(raw, 'condition_visual')),
    }
    for field in LICENSE_TYPE_FIELDS:
        fields[field] = fuzzy_match(raw.get(field), license_candidates(reference_data, state_abbrev, field), parse_confidence(raw, field))
    return {
        'state_abbrev': state_abbrev,
        'fields': fields,
        'raw_text_transcription': raw.get('raw_text_transcription', ''),
    }


## Cell 7 - Run on All Test Images

This section renders each discovered image beside a resolved field table, then stores structured results in a master list for the aggregate summary.


In [ ]:
TABLE_COLUMNS = ('field', 'source_text', 'resolved_value', 'match_score', 'vlm_conf', 'tier')


def display_value(value: Any) -> Any:
    '''Make lists and None values easier to read in notebook tables.'''
    if isinstance(value, list):
        return ', '.join(str(item) for item in value)
    if value is None:
        return ''
    return value


def tier_style(value: Any) -> str:
    colors = {
        'high': '#d9f2d9',
        'medium': '#fff3cd',
        'low': '#ffe5cc',
        'unmatched': '#f8d7da',
    }
    return f"background-color: {colors.get(value, '#ffffff')}; color: #222; font-weight: 600;"


def render_result_view(source_form: str, image_path: Path, resolved: dict[str, Any]) -> None:
    '''Display the image and resolved field table side by side.'''
    rows = []
    for field, details in resolved['fields'].items():
        rows.append({
            'field': field,
            'source_text': display_value(details['source_text']),
            'resolved_value': display_value(details['value']),
            'match_score': details['match_score'],
            'vlm_conf': round(details['vlm_conf'], 2),
            'tier': details['tier'],
        })
    table_df = pd.DataFrame(rows, columns=TABLE_COLUMNS)
    styled = table_df.style.hide(axis='index').applymap(tier_style, subset=['tier']).format({'vlm_conf': '{:.2f}', 'match_score': '{:.0f}'})

    with Image.open(image_path) as img:
        thumbnail = img.copy()
    thumbnail.thumbnail((320, 320), RESAMPLE_LANCZOS)
    thumb_buffer = BytesIO()
    thumbnail.convert('RGB').save(thumb_buffer, format='JPEG', quality=85)
    thumb_b64 = base64.b64encode(thumb_buffer.getvalue()).decode('utf-8')
    display(HTML(f'''
    <div style="display:flex; gap:24px; align-items:flex-start; margin:8px 0 12px 0;">
        <div style="min-width:340px;">
            <div style="font-weight:600; margin-bottom:8px;">{escape(source_form.title())}: {escape(image_path.name)}</div>
            <img src="data:image/jpeg;base64,{thumb_b64}" style="max-width:320px; border:1px solid #ddd; border-radius:8px;" />
        </div>
        <div style="flex:1; min-width:500px;">{styled.to_html()}</div>
    </div>
    '''))
    transcription = escape(resolved['raw_text_transcription'] or '(empty transcription)')
    display(HTML(f"<details><summary>Raw transcription</summary><pre style='white-space:pre-wrap;'>{transcription}</pre></details>"))


In [ ]:
results = []
all_images = tuple(discover_test_images())
total_cost = 0.0

print(f'Running extraction on {len(all_images)} images.')
for index, (source_form, image_path) in enumerate(all_images, start=1):
    print(f'[{index}/{len(all_images)}] {source_form}: {image_path.name}')
    try:
        extraction = extract(str(image_path))
        if 'error' in extraction:
            print(f'Error for {image_path}: {extraction["error"]}')
            results.append({
                'source_form': source_form,
                'image_path': str(image_path),
                'image_name': image_path.name,
                'error': extraction['error'],
            })
            continue
        resolved = resolve(extraction['raw'], reference_data)
        render_result_view(source_form, image_path, resolved)
        total_cost += extraction['cost_usd']
        results.append({
            'source_form': source_form,
            'image_path': str(image_path),
            'image_name': image_path.name,
            'extraction': extraction,
            'resolved': resolved,
        })
    except Exception as exc:
        print(f'Error for {image_path}: {type(exc).__name__}: {exc}')
        results.append({
            'source_form': source_form,
            'image_path': str(image_path),
            'image_name': image_path.name,
            'error': f'{type(exc).__name__}: {exc}',
        })

print(f'Completed {len(all_images)} images. Total estimated cost: ${total_cost:.4f}')


## Cell 8 - Aggregate Summary

The summary flattens the per-image results into field-level rows, then reports tier distribution, estimated spend, latency, unmatched values, and a quick unreadable-field check.


In [ ]:
if 'results' not in globals():
    print('Run Cell 7 first to populate results.')
else:
    successful_results = [result for result in results if 'error' not in result]
    if not successful_results:
        print('No successful results yet. Check your API key and re-run Cell 7.')
    else:
        flat_rows = []
        for result in successful_results:
            for field, details in result['resolved']['fields'].items():
                flat_rows.append({
                    'image_name': result['image_name'],
                    'source_form': result['source_form'],
                    'field': field,
                    'source_text': display_value(details['source_text']),
                    'resolved_value': display_value(details['value']),
                    'match_score': details['match_score'],
                    'vlm_conf': details['vlm_conf'],
                    'tier': details['tier'],
                    'is_null': not has_value(details['source_text']),
                })
        flat_df = pd.DataFrame(flat_rows)
        tier_distribution = flat_df.pivot_table(index='field', columns='tier', aggfunc='size', fill_value=0).reindex(columns=['high', 'medium', 'low', 'unmatched'], fill_value=0).sort_index()
        display(tier_distribution)

        try:
            import matplotlib.pyplot as plt
            ax = tier_distribution.plot(kind='bar', stacked=True, figsize=(12, 4), color=['#2e7d32', '#f9a825', '#ef6c00', '#c62828'])
            ax.set_title('Tier Distribution Per Field')
            ax.set_ylabel('Count')
            ax.legend(title='tier', bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            print(f'Skipped chart rendering ({type(exc).__name__}: {exc}).')

        total_cost_usd = sum(result['extraction']['cost_usd'] for result in successful_results)
        average_latency_ms = sum(result['extraction']['latency_ms'] for result in successful_results) / len(successful_results)
        summary_df = pd.DataFrame([{
            'images_processed': len(successful_results),
            'total_cost_usd': round(total_cost_usd, 6),
            'average_cost_per_image_usd': round(total_cost_usd / len(successful_results), 6),
            'average_latency_ms': round(average_latency_ms, 1),
        }])
        display(summary_df)

        unmatched_df = flat_df[(flat_df['tier'] == 'unmatched') & (flat_df['source_text'].astype(str).str.strip() != '')]
        display(unmatched_df[['image_name', 'source_form', 'field', 'source_text']].sort_values(['field', 'image_name']))

        unreadable_df = flat_df.groupby('field', as_index=False).agg(null_count=('is_null', 'sum'), total_images=('image_name', 'nunique'))
        unreadable_df['null_share'] = unreadable_df['null_count'] / unreadable_df['total_images']
        unreadable_df = unreadable_df[unreadable_df['null_count'] > unreadable_df['total_images'] / 2].sort_values(['null_share', 'field'], ascending=[False, True])
        display(unreadable_df)


## Cell 9 - Iteration Scratchpad

To iterate on the prompt or schema, re-run Cell 4 and Cell 7. To test a single image, use `extract('media/listings/<filename>')` directly after the setup cells.
